# **III. Data Collection**

Data collection will primarily utilize Reddit's PRAW (API) for datascraping. Connecting to Google drive for ease of use since the author will be using google collab for a more streamline and 'smoother' project.


In [1]:
%pip install praw numpy nltk pandas

  Using cached praw-7.8.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached prawcore-2.4.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached update_checker-0.18.0-py3-none-any.whl.metadata (2.3 kB)
Using cached praw-7.8.1-py3-none-any.whl (189 kB)
Using cached prawcore-2.4.0-py3-none-any.whl (17 kB)
Using cached update_checker-0.18.0-py3-none-any.whl (7.0 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# from dotenv import load_dotenv
# import os

# load_dotenv("/content/drive/CSCI 161/Project/redditdatascrape.env", override=True)

# CLIENT_ID = os.getenv("CLIENT_ID")
# CLIENT_SECRET = os.getenv("CLIENT_SECRET")

# print("CLIENT_ID:", CLIENT_ID)
# print("CLIENT_SECRET:", CLIENT_SECRET)


In [3]:
import pandas as pd
import numpy as np
import praw
import pandas as pd
import os

In [4]:
reddit = praw.Reddit(
    client_id="390Mv4deA5sjduaY4-n2TA",
    client_secret="-lvWJJC94wvX8-7DLNrp0wgaa70EOA",
    user_agent="script:reddit_scraper:v1.0"
)

print("Read-only mode:", reddit.read_only)

Read-only mode: True


In [10]:
subreddit = reddit.subreddit("Philippines")

posts_data = []

# Fetch more posts (adjust as needed)
for post in subreddit.search("Agriculture", limit=500):
    body = post.selftext if post.selftext else "[No text - maybe image/link post]"

    posts_data.append({
        "post_title": post.title,
        "post_url": "https://reddit.com" + post.permalink,
        "comment": None,
        "comment_author": None,
        "comment_score": None,
        "post_body": body,
        "post_score": post.score,
        "post_author": post.author.name if post.author else "[deleted]",
        "created_utc": pd.to_datetime(post.created_utc, unit="s")
    })

    # Fetch comments
    post.comments.replace_more(limit=0)
    for comment in post.comments.list():
        posts_data.append({
            "post_title": post.title,
            "post_url": "https://reddit.com" + post.permalink,
            "comment": comment.body,
            "comment_author": comment.author.name if comment.author else "[deleted]",
            "comment_score": comment.score,
            "post_body": None,
            "post_score": None,
            "post_author": None,
            "created_utc": pd.to_datetime(comment.created_utc, unit="s")
        })

# Save locally
df = pd.DataFrame(posts_data)

output_dir = "/Users/uwie/Documents/_Github/CSCI161-SocialComputing/Binwag - PhilippineRedditAgricultureAnalysis/data"
os.makedirs(output_dir, exist_ok=True) 

output_path = os.path.join(output_dir, "reddit_data.csv")
df.to_csv(output_path, index=False)


In [12]:
reddit_collected = pd.read_csv(
    "/Users/uwie/Documents/_Github/CSCI161-SocialComputing/Binwag - PhilippineRedditAgricultureAnalysis/data/reddit_data.csv"
)
reddit_collected.tail()


,post_title,post_url,comment,comment_author,comment_score,post_body,post_score,post_author,created_utc
13571,Agricultural and Biosystems Engineering,https://reddit.com/r/Philippines/comments/ckj7...,Mag aral ka kaysa reddit,grinsken,1.0,NaN,NaN,NaN,2019-08-01 10:08:10
13572,Agricultural and Biosystems Engineering,https://reddit.com/r/Philippines/comments/ckj7...,.,[deleted],-1.0,NaN,NaN,NaN,2019-08-01 06:02:11
13573,Agricultural and Biosystems Engineering,https://reddit.com/r/Philippines/comments/ckj7...,already doing that :)),inlovewithrainbows,1.0,NaN,NaN,NaN,2019-08-01 10:29:08
13574,Agricultural and Biosystems Engineering,https://reddit.com/r/Philippines/comments/ckj7...,so no tips? haha,inlovewithrainbows,1.0,NaN,NaN,NaN,2019-08-01 06:35:25
13575,‘Ambo’ damage to agriculture breaches P 1-B mark,https://reddit.com/r/Philippines/comments/gmai...,NaN,NaN,NaN,[No text - maybe image/link post],7.0,DataPatata,2020-05-18 21:20:30


In [7]:
from praw.models import Submission
import pandas as pd
import time

posts_data = []
subreddit = reddit.subreddit("Philippines")

for submission in subreddit.top(limit=None):  # or .new(limit=None)
    if "agriculture" in submission.title.lower():
        body = submission.selftext if submission.selftext else "[No text]"
        posts_data.append({
            "post_id": submission.id,
            "post_title": submission.title,
            "post_url": "https://reddit.com" + submission.permalink,
            "post_body": body,
            "post_score": submission.score,
            "post_author": submission.author.name if submission.author else "[deleted]",
            "created_utc": pd.to_datetime(submission.created_utc, unit="s")
        })
    time.sleep(0.2)

df = pd.DataFrame(posts_data)
print("✅ Posts collected:", len(df))


✅ Posts collected: 0
